In [1]:
# Cargar librerías y configurar la conexión segura
!pip install psycopg2-binary
import pandas as pd
from sqlalchemy import create_engine

db_config = {
    'user': 'practicum_student',         
    'pwd': 's65BlTKV3faNIGhmvJVzOqhs', 
    'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
    'port': 6432,              
    'db': 'data-analyst-final-project-db'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

# Aquí es donde se usa el archivo CA.pem que descargaste
engine = create_engine(connection_string, connect_args={'sslmode':'require'})
print("¡Conexión establecida con éxito con el servidor del proyecto final!")

¡Conexión establecida con éxito con el servidor del proyecto final!


In [8]:
from IPython.display import display, HTML
display(HTML("""
<style>
    /* Forzar JupyterLab moderno */
    .jp-Notebook { max-width: 100% !important; width: 100% !important; }
    .jp-Cell { max-width: 100% !important; width: 100% !important; }
    
    /* Forzar Jupyter Notebook v7+ moderno */
    .notebook-container { max-width: 100% !important; width: 100% !important; }
    .container { width: 100% !important; max-width: 100% !important; }
    .cell { max-width: 100% !important; width: 100% !important; }
</style>
"""))

In [2]:
# Imprimir las primeras 5 filas de cada tabla para inspección
for tabla in ['books', 'authors', 'publishers', 'ratings', 'reviews']:
    print(f"\n--- Primeras filas de la tabla: {tabla} ---")
    query_preview = f"SELECT * FROM {tabla} LIMIT 5;"
    df_preview = pd.io.sql.read_sql(query_preview, con=engine)
    print(df_preview)


--- Primeras filas de la tabla: books ---
   book_id  author_id                                              title  \
0        1        546                                       'Salem's Lot   
1        2        465                 1 000 Places to See Before You Die   
2        3        407  13 Little Blue Envelopes (Little Blue Envelope...   
3        4         82  1491: New Revelations of the Americas Before C...   
4        5        125                                               1776   

   num_pages publication_date  publisher_id  
0        594       2005-11-01            93  
1        992       2003-05-22           336  
2        322       2010-12-21           135  
3        541       2006-10-10           309  
4        386       2006-07-04           268  

--- Primeras filas de la tabla: authors ---
   author_id                          author
0          1                      A.S. Byatt
1          2  Aesop/Laura Harris/Laura Gibbs
2          3                 Agatha Christie

In [3]:
query_1 = """
SELECT COUNT(book_id) AS total_libros_modernos
FROM books
WHERE publication_date > '2000-01-01';
"""
df_1 = pd.io.sql.read_sql(query_1, con=engine)
print(df_1)

   total_libros_modernos
0                    819


Esta consulta nos permite cuantificar el tamaño del catálogo disponible enfocado en el siglo XXI. Nos ayuda a dimensionar si la plataforma cuenta con una oferta actualizada y atractiva para los nuevos usuarios digitales que prefieren lecturas contemporáneas.

In [4]:
query_2 = """
SELECT 
    b.book_id,
    b.title,
    COUNT(DISTINCT rev.review_id) AS conteo_reseñas,
    ROUND(AVG(rat.rating)::numeric, 2) AS calificacion_promedio
FROM books b
LEFT JOIN reviews rev ON b.book_id = rev.book_id
LEFT JOIN ratings rat ON b.book_id = rat.book_id
GROUP BY b.book_id, b.title
ORDER BY conteo_reseñas DESC;
"""
df_2 = pd.io.sql.read_sql(query_2, con=engine)
print("Mostramos los 10 más reseñados \n")
print(df_2.head(10)) 

Mostramos los 10 más reseñados 

   book_id                                              title  conteo_reseñas  \
0      948                            Twilight (Twilight  #1)               7   
1      963                                Water for Elephants               6   
2      734                                   The Glass Castle               6   
3      302  Harry Potter and the Prisoner of Azkaban (Harr...               6   
4      695  The Curious Incident of the Dog in the Night-Time               6   
5      696             The Da Vinci Code (Robert Langdon  #2)               6   
6      627                                      The Alchemist               6   
7      750                The Hobbit  or There and Back Again               6   
8      656                                     The Book Thief               6   
9      779  The Lightning Thief (Percy Jackson and the Oly...               6   

   calificacion_promedio  
0                   3.66  
1                   3

Al cruzar las tablas de calificaciones y reseñas por libro, obtenemos un ranking de popularidad y satisfacción. Los títulos situados al principio de este dataset representan los productos "estrella" que generan mayor conversación comunitaria, métrica clave para el algoritmo de recomendaciones de la app.

In [5]:
query_3 = """
SELECT 
    p.publisher,
    COUNT(b.book_id) AS total_libros_validos
FROM publishers p
INNER JOIN books b ON p.publisher_id = b.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY total_libros_validos DESC
LIMIT 1;
"""
df_3 = pd.io.sql.read_sql(query_3, con=engine)
print(df_3)

       publisher  total_libros_validos
0  Penguin Books                    42


Excluir folletos o textos cortos (menos de 50 páginas) nos revela cuál es el socio comercial (editorial) dominante en cuanto a contenido literario maduro. Identificar a este jugador líder es un insight crítico para que la startup negocie acuerdos de distribución prioritarios en su fase de lanzamiento.

In [6]:
query_4 = """
WITH libros_populares AS (
    SELECT book_id
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
)
SELECT 
    a.author,
    ROUND(AVG(r.rating)::numeric, 2) AS calificacion_promedio_autor
FROM authors a
INNER JOIN books b ON a.author_id = b.author_id
INNER JOIN ratings r ON b.book_id = r.book_id
WHERE b.book_id IN (SELECT book_id FROM libros_populares)
GROUP BY a.author
ORDER BY calificacion_promedio_autor DESC
LIMIT 1;
"""
df_4 = pd.io.sql.read_sql(query_4, con=engine)
print(df_4)

                       author  calificacion_promedio_autor
0  J.K. Rowling/Mary GrandPré                         4.29


Al aplicar el filtro estricto de mínimo 50 calificaciones, evitamos el sesgo de autores con una sola calificación perfecta de 5 estrellas. El resultado nos da al autor más aclamado y consistente por el público masivo, ideal para lanzar campañas publicitarias usándolo como estandarte del nuevo producto.

In [7]:
query_5 = """
WITH usuarios_activos AS (
    SELECT username
    FROM ratings
    GROUP BY username
    HAVING COUNT(rating_id) > 50
),
reseñas_por_usuario AS (
    SELECT 
        ua.username,
        COUNT(rev.review_id) AS total_reseñas_texto
    FROM usuarios_activos ua
    LEFT JOIN reviews rev ON ua.username = rev.username
    GROUP BY ua.username
)
SELECT ROUND(AVG(total_reseñas_texto)::numeric, 2) AS promedio_reseñas_texto_usuarios_top
FROM reseñas_por_usuario;
"""
df_5 = pd.io.sql.read_sql(query_5, con=engine)
print(df_5)

   promedio_reseñas_texto_usuarios_top
0                                24.33


Esta consulta aísla el comportamiento de los "Power Users" (usuarios que han calificado más de 50 libros) y evalúa qué tan dispuestos están a redactar una reseña escrita completa. Si el promedio es alto, indica que los usuarios más comprometidos no solo puntúan con estrellas, sino que aportan valor textual, guiando la estrategia de gamificación o recompensas de nuestra nueva aplicación.